In [29]:
# 載入作業系統模組，用於取得目前工作目錄
import os
# 載入 Python 系統模組，用於管理模組搜尋路徑
import sys
# 從 train_save 模組匯入訓練並儲存模型的核心函式
from train_save import train_and_save_model

# 取得目前工作目錄
current_dir = os.getcwd()
# 若目前目錄尚未在模組搜尋路徑中，則插入至最前面
# 確保 Python 能找到同目錄下的 train_save 模組
if current_dir not in sys.path:
    sys.path.insert(0,current_dir)

# 印出環境準備完成的提示訊息
print("環境準備完畢，已成功載入 train_and_save_model 模駔。")

環境準備完畢，已成功載入 train_and_save_model 模駔。


Part 1：定義請求與回應的 Pydantic 模型

In [30]:
# ============================================
# 定義 Pydantic 訓練相關模型（與 app.py 相同）
# ============================================

# 匯入 Pydantic 的 BaseModel 與 Field，用來定義資料模型及欄位屬性
from pydantic import BaseModel,Field
# 匯入 pprint，讓列印出的結構更容易閱讀
from pprint import pprint

# 定義「訓練請求」的資料模型
class TrainConfig(BaseModel):
    # 測試集分割比例，預設 0.2，限定範圍 0.1~0.5
    test_size: float = Field(0.2, description="測試集分割比例", ge=0.1 , le=0.5)
    # 隨機種子，預設 76，確保結果可重現，限定不可為負數
    random_state: int = Field(76, description="隨機種子", ge=0)
    # 模型演算法類型，預設 LinearRegression，可選 Lasso 或 Ridge
    model_type: str = Field("LinearRegression", description="模型演算法類型 (LinearRegression, Lasso, Ridge)")
    # 正則化強度 alpha，預設 1.0，限定範圍 0.001~100（僅 Lasso/Ridge 適用）
    alpha: float = Field(1.0, description="正則化強度 alpha (適用於 Lasso 與 Ridge)", ge= 0.001, le=100.0)

# 定義「訓練回應」的資料模型
class TrainResult(BaseModel):
    status: str = Field(..., description="執行結果狀態")        # 必填：執行結果狀態
    r2: float = Field(..., description="測試集 R-squared 決定係數")  # 必填：模型決定係數
    coef: list[float] = Field(..., description="特徵權重係數列表")    # 必填：各特徵的權重係數
    intercept: float = Field(..., description="截距")             # 必填：回歸模型的截距
    feature_coefs: dict[str, float] = Field(..., description="特徵及其權重映射")  # 必填：特徵名稱與權重的對應表
    model_type: str = Field(..., description="模型演算法類型")      # 必填：實際使用的模型演算法
    alpha: float = Field(..., description="正則化強度 alpha")       # 必填：訓練時使用的 alpha 值
    train_time: float = Field(..., description="訓練耗時 (秒)")     # 必填：訓練所花費的時間
    message:str = Field(..., description="提示訊息")               # 必填：給使用者的提示訊息

# 印出 TrainConfig 的 JSON Schema（欄位結構與限制）
print("TranConfig(BaseModel)")
pprint(TrainConfig.model_json_schema())
# 分隔線，方便區分兩個模型的輸出
print("==============================")
# 印出 TrainResult 的 JSON Schema（欄位結構與限制）
print("TrainResult(BaseModel)")
pprint(TrainResult.model_json_schema())

TranConfig(BaseModel)
{'properties': {'alpha': {'default': 1.0,
                          'description': '正則化強度 alpha (適用於 Lasso 與 Ridge)',
                          'maximum': 100.0,
                          'minimum': 0.001,
                          'title': 'Alpha',
                          'type': 'number'},
                'model_type': {'default': 'LinearRegression',
                               'description': '模型演算法類型 (LinearRegression, '
                                              'Lasso, Ridge)',
                               'title': 'Model Type',
                               'type': 'string'},
                'random_state': {'default': 76,
                                 'description': '隨機種子',
                                 'minimum': 0,
                                 'title': 'Random State',
                                 'type': 'integer'},
                'test_size': {'default': 0.2,
                              'description': '測試集分割比例',
              

拆解 train_and_save_model() 底層訓練與序列化

In [31]:
# 再次匯入訓練並儲存模型的函式（模組已於第一個 cell 載入）
from train_save import train_and_save_model
# 呼叫訓練函式：使用 Ridge 嶺迴歸，測試集比例 0.2、隨機種子 76、正則化強度 alpha=10.0
# 回傳的結果為字典（dict），儲存到 res_ridge
res_ridge:dict = train_and_save_model(
    test_size=0.2,
    random_state=76,
    model_type="Ridge",
    alpha=10.0
)
# 格式化印出 Ridge 模型的訓練結果（狀態、R²、係數、截距等）
pprint(res_ridge)

開始訓練 Ridge 嶺迴歸(α=10.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 c:\Users\User\Documents\GitHub\2027-07-03clone\backend\2026_08_07\salary_model.joblib...
模型儲存成功！
{'alpha': 10.0,
 'coef': [3.915606705818322,
          10.029103401270465,
          -1.4644383465780102,
          -1.182860911975303,
          2.1482340576072554],
 'intercept': 51.228571428571435,
 'message': 'Ridge 嶺迴歸(α=10.0) 模型訓練完成並儲存成功！',
 'model_type': 'Ridge',
 'r2': 0.8253872705107945,
 'status': 'success',
 'train_time': 0.005999326705932617}


理解 load_model_state() 全域動態更新機制

In [32]:
# 匯入 joblib，用於將模型物件序列化（儲存）與反序列化（讀取）
import joblib

# 取得目前工作目錄，作為模型檔案的存放位置
current_dir = os.getcwd()
# 組合出模型檔案的完整路徑
model_path = os.path.join(current_dir, "salary_model.joblib")
# 全域變數：存放已載入的模型、預處理器與各項中繼資料
MODEL_STATE = {}

# 定義「載入模型狀態」的函式，將模型檔案的內容讀入全域 MODEL_STATE
def load_model_state():
    global MODEL_STATE
    # 若模型檔案不存在，先訓練並儲存模型，再繼續後續載入流程
    if not os.path.exists(model_path):
        train_and_save_model()

    # 從 joblib 檔案讀取完整的模型資料字典
    model_data = joblib.load(model_path)
    # 先清空舊的全域狀態，避免殘留上一次載入的內容
    MODEL_STATE.clear()
    # 將讀取到的各項資料更新至全域 MODEL_STATE
    MODEL_STATE.update(
        {
            "model": model_data["model"],                        # 訓練好的回歸模型
            "oe": model_data["oe"],                              # 標籤編碼器（處理類別欄位）
            "ohe": model_data["ohe"],                            # 獨熱編碼器（處理類別欄位）
            "scaler": model_data["scaler"],                      # 標準化縮放器（處理數值欄位）
            "r2": model_data.get("r2"),                          # 測試集 R² 決定係數
            "feature_names": model_data["feature_names"],        # 特徵名稱列表
            "feature_coefs": model_data.get("feature_coefs",{}), # 特徵名稱與權重係數的對應表
            "model_type": model_data.get("model_type"),          # 模型演算法類型
            "alpha": model_data.get("alpha")                     # 正則化參數 alpha
        }
    )
    # 印出載入成功的提示，顯示目前模型類型與 R² Score（取小數後 4 位）
    print(f"✅ MODEL_STATE 已成功更新！當前模型：{MODEL_STATE['model_type']}，R² Score：{MODEL_STATE['r2']:.4f}")

    # 執行載入模型狀態的動作
    load_model_state()


流程串成 train_api 函數

In [33]:
from fastapi import HTTPException

def train_api(config:TrainConfig) -> dict:
    """
    訓練端點：傳入測試集比例、隨機種子、模型類型與 alpha，線上重新訓練模型，並即時更新服務所使用的模型。
    """
    try:
        # 1. 執行重新訓練並儲存模型
        res = train_and_save_model(
            test_size=config.test_size,
            random_state= config.random_state,
            model_type= config.model_type,
            alpha=config.alpha
        )
         # 2. 線上重新載入最新模型狀態至全域變數
        load_model_state()
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"線上訓練失敗: {str(e)}")

    return res

FastAPI TestClient 整合測試

In [34]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

mini_api = FastAPI()
@mini_api.post("/train", response_model=TrainResult)
def train_endpoint(config:TrainConfig):
    res = train_api(config=config)
    return res

client = TestClient(mini_api)
response = client.post("/train", json={
    "test_size": 0.2,
    "random_state": 76,
    "model_type": "Lasso",
    "alpha": 5.0
})
print("【重訓 Lasso 結果】")
print("HTTP 狀態碼:", response.status_code)
pprint(response.json())

開始訓練 Lasso 迴歸(α=5.0) (測試集比例:0.2, 隨機種子:76)....
正在將模型、預處理器與元數據序列化並儲存至 c:\Users\User\Documents\GitHub\2027-07-03clone\backend\2026_08_07\salary_model.joblib...
模型儲存成功！
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Score：0.8441
✅ MODEL_STATE 已成功更新！當前模型：Lasso，R² Sc